# Phase 2 Plot Summary Enrichment

This notebook enriches weak or short descriptions with longer plot/premise text using a conservative identifier-first process. It exists because the final mood recommender depends heavily on story text: weak descriptions lead to weak topics, weak facets, and less reliable recommendations.

Accuracy rules:

- Map IMDb `tconst` to Wikidata using property `P345`.
- Only use an English Wikipedia page linked to that exact Wikidata item.
- Keep enrichment, source, and review files separate.
- Never overwrite `data/cleaned_watch_data.csv`.
- Prefer leaving a row flagged for review over filling it with unverified or hallucinated plot text.

Final-product role: this notebook improves the text layer that later becomes BERTopic topics, emotional facets, prompt-search signals, and recommendation explanations.


## Imports, Constants, And Helper Functions

In [1]:
"""Fetch identifier-matched long plot/premise text from Wikipedia.

This script is intentionally conservative:
- It maps IMDb tconst -> Wikidata item using Wikidata property P345.
- It only uses an English Wikipedia page linked to that same Wikidata item.
- It writes separate enrichment/source/review files and never edits the
  cleaned dataset in place.
"""

from __future__ import annotations

import argparse
import json
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
from urllib.parse import quote, unquote, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup


USER_AGENT = "MovieMoodRecommenderStudentProject/1.0 (local data enrichment)"
WIKIDATA_SPARQL_URL = "https://query.wikidata.org/sparql"
WIKIPEDIA_API_URL = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_REST_SUMMARY_URL = "https://en.wikipedia.org/api/rest_v1/page/summary/{title}"
REQUEST_TIMEOUT_SECONDS = 20.0

PREFERRED_SECTION_NAMES = {
    "plot",
    "plot summary",
    "premise",
    "synopsis",
    "story",
    "storyline",
    "overview",
}

REJECT_SECTION_NAMES = {
    "cast",
    "characters",
    "production",
    "release",
    "reception",
    "awards",
    "references",
    "external links",
    "notes",
    "see also",
}


@dataclass
class EnrichedSummary:
    tconst: str
    primary_title: str
    release_year: int | str
    wikidata_id: str
    wikidata_label: str
    wikipedia_title: str
    wikipedia_url: str
    source_section: str
    enrichment_text: str
    enrichment_word_count: int
    current_description: str
    current_description_word_count: int
    match_method: str
    license_note: str


def batched(values: list[str], batch_size: int) -> Iterable[list[str]]:
    for start in range(0, len(values), batch_size):
        yield values[start : start + batch_size]


def clean_text(text: str) -> str:
    text = re.sub(r"\[[^\]]*\]", " ", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def wikipedia_title_from_url(url: str) -> str:
    path = urlparse(url).path
    title = path.rsplit("/", 1)[-1]
    return unquote(title).replace("_", " ")


def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", str(text)))


def request_json(
    session: requests.Session,
    url: str,
    *,
    params: dict | None = None,
    retries: int = 2,
    pause_seconds: float = 0.8,
    timeout_seconds: float | None = None,
) -> dict:
    last_error: Exception | None = None
    timeout_seconds = REQUEST_TIMEOUT_SECONDS if timeout_seconds is None else timeout_seconds
    for attempt in range(retries):
        try:
            response = session.get(url, params=params, timeout=timeout_seconds)
            if response.status_code == 429:
                time.sleep(pause_seconds * (attempt + 2))
                continue
            response.raise_for_status()
            return response.json()
        except Exception as exc:  # noqa: BLE001 - keep batch process resilient
            last_error = exc
            time.sleep(pause_seconds * (attempt + 1))
    raise RuntimeError(f"Request failed after retries: {url}") from last_error


def fetch_wikidata_mappings(
    session: requests.Session,
    tconsts: list[str],
    *,
    batch_size: int = 80,
    pause_seconds: float = 0.3,
) -> pd.DataFrame:
    rows: list[dict] = []

    for batch_number, batch in enumerate(batched(tconsts, batch_size), start=1):
        values = " ".join(f'"{item}"' for item in batch)
        query = f"""
SELECT ?imdb ?item ?itemLabel ?enwiki WHERE {{
  VALUES ?imdb {{ {values} }}
  ?item wdt:P345 ?imdb .
  OPTIONAL {{
    ?enwiki schema:about ?item ;
            schema:isPartOf <https://en.wikipedia.org/> .
  }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
"""
        payload = request_json(
            session,
            WIKIDATA_SPARQL_URL,
            params={"query": query, "format": "json"},
        )
        for binding in payload["results"]["bindings"]:
            item_url = binding["item"]["value"]
            rows.append(
                {
                    "tconst": binding["imdb"]["value"],
                    "wikidata_id": item_url.rsplit("/", 1)[-1],
                    "wikidata_url": item_url,
                    "wikidata_label": binding.get("itemLabel", {}).get("value", ""),
                    "enwiki_url": binding.get("enwiki", {}).get("value", ""),
                }
            )

        print(f"Wikidata batch {batch_number}: mapped {len(rows)} cumulative rows", flush=True)
        time.sleep(pause_seconds)

    if not rows:
        return pd.DataFrame(columns=["tconst", "wikidata_id", "wikidata_url", "wikidata_label", "enwiki_url"])

    mappings = pd.DataFrame(rows).drop_duplicates()
    mappings["wikipedia_title"] = mappings["enwiki_url"].map(
        lambda url: wikipedia_title_from_url(url) if isinstance(url, str) and url else ""
    )
    return mappings


def fetch_page_sections(session: requests.Session, title: str) -> list[dict]:
    payload = request_json(
        session,
        WIKIPEDIA_API_URL,
        params={
            "action": "parse",
            "page": title,
            "prop": "sections",
            "format": "json",
            "redirects": "1",
        },
    )
    return payload.get("parse", {}).get("sections", [])


def parse_section_html(session: requests.Session, title: str, section_index: str) -> str:
    payload = request_json(
        session,
        WIKIPEDIA_API_URL,
        params={
            "action": "parse",
            "page": title,
            "section": section_index,
            "prop": "text",
            "format": "json",
            "redirects": "1",
            "disableeditsection": "1",
        },
    )
    html = payload.get("parse", {}).get("text", {}).get("*", "")
    soup = BeautifulSoup(html, "html.parser")

    for unwanted in soup.select("sup, table, .mw-editsection, .reference, .noprint, style, script"):
        unwanted.decompose()

    paragraphs = []
    for paragraph in soup.find_all("p"):
        text = clean_text(paragraph.get_text(" ", strip=True))
        if text:
            paragraphs.append(text)
    return clean_text(" ".join(paragraphs))


def fetch_rest_summary(session: requests.Session, title: str) -> str:
    payload = request_json(session, WIKIPEDIA_REST_SUMMARY_URL.format(title=quote(title, safe="")))
    return clean_text(payload.get("extract", ""))


def choose_plot_section(sections: list[dict]) -> dict | None:
    for section in sections:
        line = clean_text(section.get("line", "")).lower()
        if line in PREFERRED_SECTION_NAMES:
            return section

    for section in sections:
        line = clean_text(section.get("line", "")).lower()
        if any(name in line for name in PREFERRED_SECTION_NAMES):
            return section

    return None


def enrich_row(
    session: requests.Session,
    source_row: pd.Series,
    mapping_row: pd.Series,
    *,
    min_words: int,
    pause_seconds: float,
) -> tuple[EnrichedSummary | None, dict | None]:
    title = mapping_row["wikipedia_title"]
    sections = fetch_page_sections(session, title)
    section = choose_plot_section(sections)

    source_section = ""
    enrichment_text = ""
    match_method = "wikidata_p345_to_enwiki"

    if section is not None:
        source_section = clean_text(section.get("line", ""))
        enrichment_text = parse_section_html(session, title, str(section["index"]))
        time.sleep(pause_seconds)

    if word_count(enrichment_text) < min_words:
        fallback_text = fetch_rest_summary(session, title)
        time.sleep(pause_seconds)
        if word_count(fallback_text) > word_count(enrichment_text):
            enrichment_text = fallback_text
            source_section = "REST summary fallback"
            match_method = "wikidata_p345_to_enwiki_rest_summary_fallback"

    wc = word_count(enrichment_text)
    if wc < min_words:
        return None, {
            "tconst": source_row["tconst"],
            "primary_title": source_row["primary_title"],
            "release_year": source_row["release_year"],
            "wikidata_id": mapping_row["wikidata_id"],
            "wikidata_label": mapping_row["wikidata_label"],
            "wikipedia_title": title,
            "wikipedia_url": mapping_row["enwiki_url"],
            "reason": f"enrichment_text_too_short_{wc}_words",
            "current_description": source_row["description"],
        }

    enriched = EnrichedSummary(
        tconst=source_row["tconst"],
        primary_title=source_row["primary_title"],
        release_year=source_row["release_year"],
        wikidata_id=mapping_row["wikidata_id"],
        wikidata_label=mapping_row["wikidata_label"],
        wikipedia_title=title,
        wikipedia_url=mapping_row["enwiki_url"],
        source_section=source_section,
        enrichment_text=enrichment_text,
        enrichment_word_count=wc,
        current_description=source_row["description"],
        current_description_word_count=word_count(source_row["description"]),
        match_method=match_method,
        license_note="Wikipedia text is available under CC BY-SA; keep source attribution if used downstream.",
    )
    return enriched, None

## Notebook Runner

Use `run_enrichment(...)` instead of command-line arguments. The defaults write canonical Phase 2 files under `data/`.

In [2]:
def run_enrichment(
    input_path="data/cleaned_watch_data.csv",
    output_path="data/phase2_long_plot_summaries.csv",
    sources_output_path="data/phase2_wikidata_wikipedia_sources.csv",
    review_output_path="data/phase2_enrichment_needs_review.csv",
    limit=None,
    min_current_description_words=25,
    min_enrichment_words=60,
    pause_seconds=0.05,
    request_timeout_seconds=20,
    checkpoint_every=25,
    reuse_sources=True,
    resume=True,
    retry_fetch_errors=True,
):
    global REQUEST_TIMEOUT_SECONDS
    REQUEST_TIMEOUT_SECONDS = request_timeout_seconds

    df = pd.read_csv(input_path).reset_index(drop=True)
    if limit:
        df = df.head(limit).copy()

    current_words = df["description"].fillna("").map(word_count)
    candidates = df[current_words <= min_current_description_words].copy()
    print(f"Loaded {len(df)} rows; enrichment candidates: {len(candidates)}")

    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    source_path = Path(sources_output_path)
    if reuse_sources and source_path.exists():
        mappings = pd.read_csv(source_path)
        mappings = candidates[
            [
                "tconst", "primary_title", "original_title", "display_title",
                "release_year", "genres", "description",
            ]
        ].merge(
            mappings[
                [
                    "tconst", "wikidata_id", "wikidata_url", "wikidata_label",
                    "enwiki_url", "wikipedia_title",
                ]
            ].drop_duplicates("tconst"),
            on="tconst",
            how="left",
        )
        print(f"Reused source mapping: {sources_output_path}", flush=True)
    else:
        mappings = fetch_wikidata_mappings(session, candidates["tconst"].dropna().astype(str).tolist())
        mappings = mappings.merge(
            candidates[
                [
                    "tconst", "primary_title", "original_title", "display_title",
                    "release_year", "genres", "description",
                ]
            ],
            on="tconst",
            how="right",
        )
        mappings.to_csv(sources_output_path, index=False)
        print(f"Wrote source mapping: {sources_output_path}", flush=True)

    mappings["has_enwiki"] = mappings["enwiki_url"].fillna("").ne("")
    print(f"Rows with English Wikipedia page via P345: {int(mappings['has_enwiki'].sum())}")

    enriched_rows = []
    review_rows = []
    completed_tconsts = set()

    output_file = Path(output_path)
    review_file = Path(review_output_path)

    if resume and output_file.exists():
        existing_enriched = pd.read_csv(output_file)
        if not existing_enriched.empty:
            enriched_rows = [EnrichedSummary(**row) for row in existing_enriched.to_dict(orient="records")]
            completed_tconsts.update(existing_enriched["tconst"].astype(str))
        print(f"Resuming with {len(enriched_rows)} existing enriched rows", flush=True)

    if resume and review_file.exists():
        existing_review = pd.read_csv(review_file)
        if not existing_review.empty:
            if retry_fetch_errors and "reason" in existing_review.columns:
                existing_review = existing_review[
                    ~existing_review["reason"].fillna("").astype(str).str.startswith("fetch_error:")
                ].copy()
            review_rows = existing_review.to_dict(orient="records")
            completed_tconsts.update(existing_review["tconst"].astype(str))
        print(f"Resuming with {len(review_rows)} existing review rows", flush=True)

    no_mapping = mappings[~mappings["has_enwiki"]]
    for _, row in no_mapping.iterrows():
        if str(row["tconst"]) in completed_tconsts:
            continue
        review_rows.append({
            "tconst": row["tconst"],
            "primary_title": row["primary_title"],
            "release_year": row["release_year"],
            "wikidata_id": row.get("wikidata_id", ""),
            "wikidata_label": row.get("wikidata_label", ""),
            "wikipedia_title": row.get("wikipedia_title", ""),
            "wikipedia_url": row.get("enwiki_url", ""),
            "reason": "no_english_wikipedia_page_matched_by_tconst",
            "current_description": row["description"],
        })
        completed_tconsts.add(str(row["tconst"]))

    mapped = mappings[mappings["has_enwiki"]].copy()
    source_by_tconst = candidates.set_index("tconst", drop=False)

    for number, (_, mapping_row) in enumerate(mapped.iterrows(), start=1):
        tconst = mapping_row["tconst"]
        if str(tconst) in completed_tconsts:
            continue
        source_row = source_by_tconst.loc[tconst]
        try:
            enriched, review = enrich_row(
                session,
                source_row,
                mapping_row,
                min_words=min_enrichment_words,
                pause_seconds=pause_seconds,
            )
            if enriched is not None:
                enriched_rows.append(enriched)
            if review is not None:
                review_rows.append(review)
            completed_tconsts.add(str(tconst))
        except Exception as exc:
            review_rows.append({
                "tconst": source_row["tconst"],
                "primary_title": source_row["primary_title"],
                "release_year": source_row["release_year"],
                "wikidata_id": mapping_row.get("wikidata_id", ""),
                "wikidata_label": mapping_row.get("wikidata_label", ""),
                "wikipedia_title": mapping_row.get("wikipedia_title", ""),
                "wikipedia_url": mapping_row.get("enwiki_url", ""),
                "reason": f"fetch_error: {type(exc).__name__}: {exc}",
                "current_description": source_row["description"],
            })
            completed_tconsts.add(str(tconst))

        if number % checkpoint_every == 0:
            pd.DataFrame([row.__dict__ for row in enriched_rows]).to_csv(output_path, index=False)
            pd.DataFrame(review_rows).to_csv(review_output_path, index=False)
            print(
                f"Wikipedia pages checked: {number}/{len(mapped)}; "
                f"enriched: {len(enriched_rows)}; review: {len(review_rows)}; checkpoint written",
                flush=True,
            )

    pd.DataFrame([row.__dict__ for row in enriched_rows]).to_csv(output_path, index=False)
    pd.DataFrame(review_rows).to_csv(review_output_path, index=False)

    print(f"Wrote enrichments: {output_path} ({len(enriched_rows)} rows)")
    print(f"Wrote review file: {review_output_path} ({len(review_rows)} rows)")
    return pd.DataFrame([row.__dict__ for row in enriched_rows]), pd.DataFrame(review_rows)


## Example Runs

These are intentionally commented so opening/executing the notebook does not accidentally start a long internet crawl.

In [3]:
# Small smoke test:
# enriched_df, review_df = run_enrichment(
#     limit=30,
#     min_current_description_words=25,
#     output_path="data/phase2_smoke_long_plot_summaries.csv",
#     review_output_path="data/phase2_smoke_enrichment_needs_review.csv",
#     reuse_sources=True,
#     resume=True,
# )

# Priority batch for weak descriptions:
# enriched_df, review_df = run_enrichment(
#     min_current_description_words=25,
#     output_path="data/phase2_long_plot_summaries.csv",
#     review_output_path="data/phase2_enrichment_needs_review.csv",
#     reuse_sources=True,
#     resume=True,
#     retry_fetch_errors=True,
# )


## Current Canonical Phase 2 Outputs

In [4]:
for path in [
    "data/phase2_enriched_summaries_combined.csv",
    "data/phase2_enrichment_needs_review.csv",
    "data/phase2_wikidata_wikipedia_sources.csv",
    "data/cleaned_watch_data_phase2_enriched.csv",
]:
    p = Path(path)
    if p.exists():
        df = pd.read_csv(p)
        print(path, df.shape)
    else:
        print(path, "missing")


data/phase2_enriched_summaries_combined.csv (224, 15)
data/phase2_enrichment_needs_review.csv (414, 10)
data/phase2_wikidata_wikipedia_sources.csv (7861, 13)
data/cleaned_watch_data_phase2_enriched.csv (7848, 21)
